In [0]:
df_products = spark.table("ecommerce_dev.bronze.products")

df_products.printSchema()
df_products.limit(5).display()
print(f"Row count: {df_products.count()}")

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_table: string (nullable = true)



product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,_ingested_at,_source_file,_source_table
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products


Row count: 32951


In [0]:
from pyspark.sql.functions import *

# Nulls per column — products table is known to have missing category names and dimensions
df_products.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_products.columns]
).display()

# Full-row duplicates
print("Full duplicate rows:", df_products.count() - df_products.dropDuplicates().count())

# product_id should be unique — PK
print("Duplicate product_id:", df_products.count() - df_products.dropDuplicates(["product_id"]).count())

# check _rescued_data
df_products.select("_rescued_data").filter(col("_rescued_data").isNotNull()).display()

# distinct category count and a peek
df_products.select("product_category_name").distinct().count()
df_products.groupBy("product_category_name").count().orderBy(desc("count")).limit(15).display()

# numeric sanity on dimensions/weight
df_products.select(
    min("product_weight_g"), max("product_weight_g"),
    min("product_length_cm"), max("product_length_cm"),
    min("product_height_cm"), max("product_height_cm"),
    min("product_width_cm"), max("product_width_cm")
).display()

print("Rows with weight <= 0:", df_products.filter(col("product_weight_g") <= 0).count())

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,_ingested_at,_source_file,_source_table
0,610,610,610,610,2,2,2,2,32951,0,0,0


Full duplicate rows: 0
Duplicate product_id: 0


_rescued_data


product_category_name,count
cama_mesa_banho,3029
esporte_lazer,2867
moveis_decoracao,2657
beleza_saude,2444
utilidades_domesticas,2335
automotivo,1900
informatica_acessorios,1639
brinquedos,1411
relogios_presentes,1329
telefonia,1134


min(product_weight_g),max(product_weight_g),min(product_length_cm),max(product_length_cm),min(product_height_cm),max(product_height_cm),min(product_width_cm),max(product_width_cm)
0,40425,7,105,2,105,6,118


Rows with weight <= 0: 4


In [0]:
from pyspark.sql.functions import *

# confirm the 610 nulls are the same rows
df_products.filter(
    col("product_category_name").isNull() &
    col("product_name_lenght").isNull() &
    col("product_description_lenght").isNull() &
    col("product_photos_qty").isNull()
).count()

# look at the 2 rows missing dimensions
df_products.filter(col("product_weight_g").isNull()).display()

# look at the 4 rows with weight = 0
df_products.filter(col("product_weight_g") == 0).display()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,_ingested_at,_source_file,_source_table
09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,null,null,null,null,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
5eb564652db742ff8f28759cd8d2652a,null,null,null,null,null,null,null,null,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,_ingested_at,_source_file,_source_table
81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51,529,1,0,30,25,30,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48,528,1,0,30,25,30,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53,528,1,0,30,25,30,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products
e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53,528,1,0,30,25,30,null,2026-08-04T00:55:54.392Z,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv,products


In [0]:
from pyspark.sql.functions import *

df_silver_products = (
    df_products
    # fix source column-name typos: lenght -> length
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    # fill missing category with explicit 'unknown' — better than null for Gold-layer joins
    .withColumn("product_category_name",
                when(col("product_category_name").isNull(), "unknown")
                .otherwise(col("product_category_name")))
    # treat weight=0 as a data quality issue -> convert to null (0g is not physically valid)
    .withColumn("product_weight_g",
                when(col("product_weight_g") == 0, None)
                .otherwise(col("product_weight_g")))
    # add a QC flag column so Gold/analysis layer can filter these out explicitly if needed
    .withColumn("has_missing_dimensions",
                col("product_weight_g").isNull() |
                col("product_length_cm").isNull() |
                col("product_height_cm").isNull() |
                col("product_width_cm").isNull())
    .drop("_rescued_data", "_source_file", "_source_table")
    .withColumnRenamed("_ingested_at", "bronze_ingested_at")
)

df_silver_products.limit(5).display()
print("Products with missing dimensions:", df_silver_products.filter(col("has_missing_dimensions")).count())

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,bronze_ingested_at,has_missing_dimensions
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-08-04T00:55:54.392Z,false
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-08-04T00:55:54.392Z,false
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-08-04T00:55:54.392Z,false
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-08-04T00:55:54.392Z,false
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-08-04T00:55:54.392Z,false


Products with missing dimensions: 6


In [0]:
(df_silver_products.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("ecommerce_dev.silver.products"))

In [0]:
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.products
    ALTER COLUMN product_id SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.products
    ADD CONSTRAINT pk_product_id PRIMARY KEY (product_id)
""")

DataFrame[]

In [0]:
spark.sql("""
    COMMENT ON TABLE ecommerce_dev.silver.products IS
    'Cleaned product dimension. Fixed source typo (lenght->length). Missing category_name filled as unknown. weight_g=0 treated as invalid and nulled (not a real measurement). has_missing_dimensions flag added for downstream filtering. PK: product_id.'
""")

DataFrame[]